In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: Script en Python para el calculo de FAP para indices climaticos a partir del RR. 
# FUNCIONA PARA PM10 Y PM2.5
# Periodo: 2000-2019
# ==========================================

In [ ]:
import pandas as pd
import numpy as np
import os
import math

# CONFIGURAR ARCHIVOS Y COLUMNAS

archivo_entrada = "AGREGAR RUTA DEL ARCHIVO (RR)"

# Carpeta donde está el archivo de entrada
carpeta = os.path.dirname(archivo_entrada)

# Archivo de salida en la misma carpeta
archivo_salida = os.path.join(carpeta, "NOMBRE DE SALIDA.csv")

# Columnas que definen cada bloque
col_enfermedad = "Enfermedad"
col_anio = "Anio"
col_sexo = "Sexo"
col_edad = "Edad_gpo"
col_indice = "Indice"

# Columnas necesarias para el cálculo
col_n = "nx"
col_rr = "RR"


# LEER ARCHIVO

df = pd.read_csv(archivo_entrada)

# Limpiar nombres de columnas
df.columns = df.columns.str.strip()

# Convertir columnas relevantes a numérico
df[col_anio] = pd.to_numeric(df[col_anio], errors="coerce")
df[col_n] = pd.to_numeric(df[col_n], errors="coerce")
df[col_rr] = pd.to_numeric(df[col_rr], errors="coerce")

# Eliminar filas sin información clave
df = df.dropna(subset=[
    col_enfermedad,
    col_anio,
    col_sexo,
    col_edad,
    col_indice,
    col_n,
    col_rr
]).copy()


# CREAR ID DE BLOQUE

columnas_bloque = [
    col_enfermedad,
    col_anio,
    col_sexo,
    col_edad,
    col_indice
]

cambio_bloque = (df[columnas_bloque] != df[columnas_bloque].shift(1)).any(axis=1)
df["bloque_id"] = cambio_bloque.cumsum()

# FUNCIÓN PARA CALCULAR FAP

def calcular_fap_bloque(grupo):
    grupo = grupo.copy()

    N = grupo[col_n].sum()

    if pd.isna(N) or N == 0:
        return pd.Series({
            col_enfermedad: grupo[col_enfermedad].iloc[0],
            col_anio: grupo[col_anio].iloc[0],
            col_sexo: grupo[col_sexo].iloc[0],
            col_edad: grupo[col_edad].iloc[0],
            col_indice: grupo[col_indice].iloc[0],
            "nrows": len(grupo),
            "FAP_%": np.nan
        })

    # Proporciones
    grupo["p_i"] = grupo[col_n] / N

    # Fórmula
    numerador = (grupo["p_i"] * (grupo[col_rr] - 1)).sum()
    denominador = (grupo["p_i"] * grupo[col_rr]).sum()

    if denominador == 0:
        fap = np.nan
    else:
        fap = (numerador / denominador) * 100

    # Si es negativo, igualar a 0
    if pd.notna(fap) and fap < 0:
        fap = 0
        
    # Redondeo personalizado
    if pd.notna(fap):
        entero = math.floor(fap)
        decimal = fap - entero
        
        if decimal > 0.4:
            fap = entero + 1
        else:
            fap = entero

    return pd.Series({
        col_enfermedad: grupo[col_enfermedad].iloc[0],
        col_anio: grupo[col_anio].iloc[0],
        col_sexo: grupo[col_sexo].iloc[0],
        col_edad: grupo[col_edad].iloc[0],
        col_indice: grupo[col_indice].iloc[0],
        "nrows": len(grupo),
        "FAP_%": fap
    })


# CALCULAR TODOS LOS BLOQUES

resultado = (
    df.groupby("bloque_id", as_index=False)
      .apply(calcular_fap_bloque)
      .reset_index(drop=True)
)


# GUARDAR CSV FINAL

resultado.to_csv(archivo_salida, index=False)


# MOSTRAR RESULTADO

print("Archivo generado correctamente:")
print(archivo_salida)

print("\nPrimeras filas:")
print(resultado.head(20))